In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
from sklearn.preprocessing import OneHotEncoder #import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error



# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)



In [ ]:
# Task 1: Write your code here:
# Load the CSV file
Q1_data_path = os.path.join(path, 'Q1_data.csv')
df_Q1data = pd.read_csv(Q1_data_path)

print(f"Shape: {df_Q1data.shape}")


In [ ]:
# Task 2: Write your code here:
df_Q1data.head()

In [ ]:
# Task 3: Write your code here:
df_Q1data.info()

In [ ]:
# Task 4: Write your code here:
df_Q1data.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df_Q1data['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery_Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_Q1data = df_Q1data.drop(['Order_ID'], axis=1)





In [ ]:
# Task 2: Write your code here:
# 2. Do we have missing values?
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df_Q1data)

df_Q1data = df_Q1data.fillna(df_Q1data.mean())

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_Q1data)

In [ ]:
# Task 4: Write your code here:

categorical_cols = df_Q1data.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    # TODO: Apply fit_transform to encode the column
    df_Q1data[col] = le.fit_transform(df_Q1data[col])

df_Q1data.head()




In [ ]:
# Task 5: Write your code here:


numerical_cols =  df_Q1data.select_dtypes(include=["number"]).columns.drop("Delivery_Time")

scaler = StandardScaler()

# Apply fit_transform to scale the numerical columns
df_Q1data[numerical_cols] = scaler.fit_transform(df_Q1data[numerical_cols])

df_Q1data.head()


In [ ]:
# Task 6: Write your code here:
 # Is the target imbalanced? no
def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()
  plt.show()

check_target_imbalance(df_Q1data, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
X = df_Q1data.drop("Delivery_Time", axis=1).astype(float)
y = df_Q1data['Delivery_Time']


In [ ]:
# Task 2,3,4,5: Write your code here:

# use Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Model trained!")

y_pred = model.predict(X_train)

mae = mean_absolute_error(y_test, y_pred)

print(f"MAE:  ${mae:,.2f}")
print(f"MAE:  ${mae.mean():,.2f}")


In [ ]:
# Task 1: Write your code here:

feature_cols=['Distance_km','Weather','Traffic_Level','Time_of_Day']
# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 6))
plt.hist(y_pred, bins=30, edgecolor='black')
plt.title('Distribution of Predictions')
plt.xlabel('Predicted Delivery_Time')
plt.ylabel('Count')
plt.show()

In [ ]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [ ]:
models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
  "CatBoost": CatBoostRegressor(verbose=0)
}

In [ ]:
all_results = {}

for name in models:
  all_results[name] = {'mae': [], 'rmse': [], 'r2': []}

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)
    all_results[model_name]["mae"].append(mae)
    all_results[model_name]["rmse"].append(rmse)
    all_results[model_name]["r2"].append(r2)




